# Batch Normalization vs Layer Normalization (from scratch with NumPy)

**Objective:** Understand how and why normalizing data improves neural network training, by implementing **Batch Normalization** and **Layer Normalization** manually using NumPy — no deep learning framework.

**Tasks covered:**
1. Create a small dataset using a NumPy array.
2. Implement **Batch Normalization** — mean, variance, and normalized values per **feature (column)**.
3. Apply learnable scale (**gamma**) and shift (**beta**) parameters to the normalized output.
4. Implement **Layer Normalization** — mean, variance, and normalized values per **sample (row)**.
5. Compare the outputs of Batch Normalization and Layer Normalization.

**Requirements:**
```
pip install numpy pandas
```


## 1. Create a Small Dataset

We'll simulate a mini-batch of **4 samples**, each with **3 features** — like 4 rows going through a neural network layer, where each column is a feature/neuron activation.

In [1]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=4, suppress=True)

# 4 samples (rows) x 3 features (columns)
X = np.array([
    [ 2.0,  10.0, -1.0],
    [ 4.0,  12.0,  0.5],
    [ 6.0,   8.0,  2.0],
    [ 8.0,  14.0,  1.5]
])

print("Dataset X (shape = samples x features):")
pd.DataFrame(X, columns=['feature_1', 'feature_2', 'feature_3'],
             index=[f"sample_{i}" for i in range(X.shape[0])])

Dataset X (shape = samples x features):


,feature_1,feature_2,feature_3
sample_0,2.0,10.0,-1.0
sample_1,4.0,12.0,0.5
sample_2,6.0,8.0,2.0
sample_3,8.0,14.0,1.5


## 2. Batch Normalization (BatchNorm)

**Batch Normalization normalizes across the batch dimension — i.e. per *feature* (column).**

For each feature/column $j$, across all samples in the batch:

$$\mu_j = \frac{1}{N}\sum_{i=1}^{N} x_{i,j} \qquad
\sigma_j^2 = \frac{1}{N}\sum_{i=1}^{N} (x_{i,j} - \mu_j)^2$$

$$\hat{x}_{i,j} = \frac{x_{i,j} - \mu_j}{\sqrt{\sigma_j^2 + \epsilon}}$$

This means: each column ends up with mean ≈ 0 and variance ≈ 1, computed **down** the rows.

In [2]:
epsilon = 1e-8

def batch_norm(X, epsilon=1e-8):
    """
    Normalize per feature (column) -- across the batch (rows).
    """
    mean = X.mean(axis=0)          # mean per column -> shape (features,)
    var = X.var(axis=0)            # variance per column -> shape (features,)
    X_norm = (X - mean) / np.sqrt(var + epsilon)
    return X_norm, mean, var

bn_output, bn_mean, bn_var = batch_norm(X, epsilon)

print("Per-feature mean (mu):", bn_mean)
print("Per-feature variance (sigma^2):", bn_var)
print("\nBatch-normalized output:")
pd.DataFrame(bn_output, columns=['feature_1', 'feature_2', 'feature_3'],
             index=[f"sample_{i}" for i in range(X.shape[0])])

Per-feature mean (mu): [ 5.   11.    0.75]
Per-feature variance (sigma^2): [5.     5.     1.3125]

Batch-normalized output:


,feature_1,feature_2,feature_3
sample_0,-1.341641,-0.447214,-1.527525
sample_1,-0.447214,0.447214,-0.218218
sample_2,0.447214,-1.341641,1.091089
sample_3,1.341641,1.341641,0.654654


In [3]:
# Sanity check: each COLUMN should now have mean ~ 0 and variance ~ 1
print("Column means after BatchNorm:", bn_output.mean(axis=0))
print("Column variances after BatchNorm:", bn_output.var(axis=0))

Column means after BatchNorm: [ 0.  0. -0.]
Column variances after BatchNorm: [1. 1. 1.]


## 3. Applying Gamma (scale) and Beta (shift)

Pure normalization forces every feature to mean 0 / variance 1, which can limit what the network can represent. So BatchNorm adds two **learnable parameters per feature**:

$$y_{i,j} = \gamma_j \cdot \hat{x}_{i,j} + \beta_j$$

- $\gamma$ (gamma) rescales the normalized values (learned "how much spread to allow")
- $\beta$ (beta) shifts them (learned "what the new mean should be")

During real training these are learned via backpropagation. Here we just pick example values to demonstrate the transformation.

In [4]:
# Example learnable parameters, one value per feature
gamma = np.array([1.5, 0.5, 2.0])   # scale
beta  = np.array([0.0, 1.0, -0.5])  # shift

def apply_scale_shift(X_norm, gamma, beta):
    return gamma * X_norm + beta

bn_final_output = apply_scale_shift(bn_output, gamma, beta)

print("gamma:", gamma)
print("beta :", beta)
print("\nFinal Batch Normalization output (after gamma & beta):")
pd.DataFrame(bn_final_output, columns=['feature_1', 'feature_2', 'feature_3'],
             index=[f"sample_{i}" for i in range(X.shape[0])])

gamma: [1.5 0.5 2. ]
beta : [ 0.   1.  -0.5]

Final Batch Normalization output (after gamma & beta):


,feature_1,feature_2,feature_3
sample_0,-2.012461,0.776393,-3.555050
sample_1,-0.670820,1.223607,-0.936436
sample_2,0.670820,0.329180,1.682179
sample_3,2.012461,1.670820,0.809307


> Note: with $\gamma=1$ and $\beta=0$ for every feature, this reduces exactly to plain normalization. Non-trivial gamma/beta let the network "undo" normalization if that's what minimizes the loss, while still keeping the benefits of stable gradients during training.

## 4. Layer Normalization (LayerNorm)

**Layer Normalization normalizes across the feature dimension — i.e. per *sample* (row)**, instead of per feature.

For each sample/row $i$, across all its features:

$$\mu_i = \frac{1}{D}\sum_{j=1}^{D} x_{i,j} \qquad
\sigma_i^2 = \frac{1}{D}\sum_{j=1}^{D} (x_{i,j} - \mu_i)^2$$

$$\hat{x}_{i,j} = \frac{x_{i,j} - \mu_i}{\sqrt{\sigma_i^2 + \epsilon}}$$

This means: each **row** ends up with mean ≈ 0 and variance ≈ 1, computed **across** the columns. Unlike BatchNorm, this doesn't depend on other samples in the batch at all — which is why LayerNorm works well with batch size 1 and is standard in Transformers/RNNs.

In [5]:
def layer_norm(X, epsilon=1e-8):
    """
    Normalize per sample (row) -- across the features (columns).
    """
    mean = X.mean(axis=1, keepdims=True)   # mean per row -> shape (samples, 1)
    var = X.var(axis=1, keepdims=True)     # variance per row -> shape (samples, 1)
    X_norm = (X - mean) / np.sqrt(var + epsilon)
    return X_norm, mean.flatten(), var.flatten()

ln_output, ln_mean, ln_var = layer_norm(X, epsilon)

print("Per-sample mean (mu):", ln_mean)
print("Per-sample variance (sigma^2):", ln_var)
print("\nLayer-normalized output:")
pd.DataFrame(ln_output, columns=['feature_1', 'feature_2', 'feature_3'],
             index=[f"sample_{i}" for i in range(X.shape[0])])

Per-sample mean (mu): [3.6667 5.5    5.3333 7.8333]
Per-sample variance (sigma^2): [21.5556 23.1667  6.2222 26.0556]

Layer-normalized output:


,feature_1,feature_2,feature_3
sample_0,-0.358979,1.364121,-1.005141
sample_1,-0.311645,1.350460,-1.038815
sample_2,0.267261,1.069045,-1.336306
sample_3,0.032651,1.208093,-1.240744


In [6]:
# Sanity check: each ROW should now have mean ~ 0 and variance ~ 1
print("Row means after LayerNorm:", ln_output.mean(axis=1))
print("Row variances after LayerNorm:", ln_output.var(axis=1))

Row means after LayerNorm: [0. 0. 0. 0.]
Row variances after LayerNorm: [1. 1. 1. 1.]


In [7]:
# LayerNorm also has its own learnable gamma/beta -- one value PER FEATURE,
# applied identically to every sample (same as BatchNorm's formula, different normalization axis)
ln_gamma = np.array([1.0, 1.0, 1.0])
ln_beta  = np.array([0.0, 0.0, 0.0])

ln_final_output = apply_scale_shift(ln_output, ln_gamma, ln_beta)

print("Final Layer Normalization output (gamma=1, beta=0 -> identical to ln_output here):")
pd.DataFrame(ln_final_output, columns=['feature_1', 'feature_2', 'feature_3'],
             index=[f"sample_{i}" for i in range(X.shape[0])])

Final Layer Normalization output (gamma=1, beta=0 -> identical to ln_output here):


,feature_1,feature_2,feature_3
sample_0,-0.358979,1.364121,-1.005141
sample_1,-0.311645,1.350460,-1.038815
sample_2,0.267261,1.069045,-1.336306
sample_3,0.032651,1.208093,-1.240744


## 5. Comparing Batch Normalization vs Layer Normalization

In [8]:
comparison = pd.concat(
    {
        "Original": pd.DataFrame(X, columns=['f1', 'f2', 'f3']),
        "BatchNorm (normalized)": pd.DataFrame(bn_output, columns=['f1', 'f2', 'f3']),
        "BatchNorm (gamma/beta applied)": pd.DataFrame(bn_final_output, columns=['f1', 'f2', 'f3']),
        "LayerNorm (normalized)": pd.DataFrame(ln_output, columns=['f1', 'f2', 'f3']),
    },
    axis=1
)
comparison.index = [f"sample_{i}" for i in range(X.shape[0])]
comparison

Original            BatchNorm (normalized)                      \
               f1    f2   f3                     f1        f2        f3   
sample_0      2.0  10.0 -1.0              -1.341641 -0.447214 -1.527525   
sample_1      4.0  12.0  0.5              -0.447214  0.447214 -0.218218   
sample_2      6.0   8.0  2.0               0.447214 -1.341641  1.091089   
sample_3      8.0  14.0  1.5               1.341641  1.341641  0.654654   

         BatchNorm (gamma/beta applied)                      \
                                     f1        f2        f3   
sample_0                      -2.012461  0.776393 -3.555050   
sample_1                      -0.670820  1.223607 -0.936436   
sample_2                       0.670820  0.329180  1.682179   
sample_3                       2.012461  1.670820  0.809307   

         LayerNorm (normalized)                      
                             f1        f2        f3  
sample_0              -0.358979  1.364121 -1.005141  
sample_1              -0.311645  1.350460 -1.038815  
sample_2               0.267261  1.069045 -1.336306  
sample_3               0.032651  1.208093 -1.240744

In [9]:
print("=" * 60)
print("STATISTICS COMPARISON")
print("=" * 60)

print("\nBatchNorm -- computed per COLUMN (feature), across rows:")
print(f"  Column means : {bn_output.mean(axis=0)}")
print(f"  Column vars  : {bn_output.var(axis=0)}")
print(f"  Row means    : {bn_output.mean(axis=1)}   <- NOT necessarily 0")

print("\nLayerNorm -- computed per ROW (sample), across columns:")
print(f"  Row means    : {ln_output.mean(axis=1)}")
print(f"  Row vars     : {ln_output.var(axis=1)}")
print(f"  Column means : {ln_output.mean(axis=0)}   <- NOT necessarily 0")

STATISTICS COMPARISON

BatchNorm -- computed per COLUMN (feature), across rows:
  Column means : [ 0.  0. -0.]
  Column vars  : [1. 1. 1.]
  Row means    : [-1.1055 -0.0727  0.0656  1.1126]   <- NOT necessarily 0

LayerNorm -- computed per ROW (sample), across columns:
  Row means    : [0. 0. 0. 0.]
  Row vars     : [1. 1. 1. 1.]
  Column means : [-0.0927  1.2479 -1.1553]   <- NOT necessarily 0


### Key Differences

| Aspect | Batch Normalization | Layer Normalization |
|---|---|---|
| Normalizes across | The **batch** (all samples), per feature | The **features**, per sample |
| Axis used in NumPy | `axis=0` (down columns) | `axis=1` (across rows) |
| Guaranteed mean≈0, var≈1 | Every **column** | Every **row** |
| Depends on other samples in the batch? | **Yes** — statistics computed across the batch | **No** — each sample normalized independently |
| Works with batch size = 1? | No (variance across 1 sample is meaningless) | Yes |
| Common use case | CNNs, standard feedforward nets with large batches | RNNs, Transformers, variable/small batch sizes |
| Learnable params | gamma, beta — one pair **per feature**, shared across samples | gamma, beta — one pair **per feature**, shared across samples (but stats computed per sample) |

### Takeaway

Both techniques rescale activations to have stable mean and variance, which helps gradients flow better and lets networks train faster and more reliably. The core difference is **which axis the statistics are computed over**:
- BatchNorm looks **down a column** (across the batch) — so it needs a reasonably large, representative batch.
- LayerNorm looks **across a row** (across the features of a single sample) — so it works the same regardless of batch size, which is why it's the default choice in Transformer architectures.